# Configuration

In [1]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [2]:
from scipy.stats import norm

# estimators
from econml.grf import CausalForest

In [3]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Main

In [50]:
treatment_effect_meta = {
    'exp1': {
        'mean': 1,  # mean of the exponential distribution
        'sample': np.random.exponential(scale=1, size=100), 
        'cdf': lambda x: 1 - np.exp(- x),  # cumulative distribution function
    }, 
    'exp2': {
        'mean': 2,  # mean of the exponential distribution
        'sample': np.random.exponential(scale=2, size=500),
        'cdf': lambda x: 1 - np.exp(- x / 2),  # cumulative distribution function
    }, 
    'normal1': {
        'mean': 3, 
        'std': 2, 
        'sample': np.random.normal(loc=3, scale=2, size=200), 
        'cdf': lambda x: norm.cdf(x, loc=3, scale=2),  # cumulative distribution function
    }, 
    'normal': {
        'mean': 2, 
        'std': 5, 
        'sample': np.random.normal(loc=2, scale=5, size=300), 
        'cdf': lambda x: norm.cdf(x, loc=2, scale=5),  # cumulative distribution function
    }
}

target_variable_mean = np.array([
    max(
        np.random.exponential(scale=1), np.random.exponential(scale=2), np.random.normal(loc=3, scale=2), 
        np.random.normal(loc=2, scale=5)
    )
    for _ in range(int(1e5))
]).mean()

data = {key: value['sample'] for key, value in treatment_effect_meta.items()}

cdf = lambda x: np.prod([dstn['cdf'](x) for dstn in treatment_effect_meta.values()])

In [51]:
# naive estimator
def naive_estimator(data: dict):
    """ 
    maximum over the average of the variables
    """
    return max([sample.mean() for sample in data.values()])

# bootstrap estimator
def bootstrap_estimator(data: dict, num_bootstraps: int = 5000):
    """
    bootstrap the estimator
    """
    # bootstrapping
    bootstraps = pd.DataFrame({key: np.random.choice(value, size=num_bootstraps) for key, value in data.items()})
    return bootstraps.max(axis=1).mean()


In [52]:
print(f"Naive estimator: {naive_estimator(data)}")
print(f"Bootstrap estimator: {bootstrap_estimator(data)}")
print('True value: ', target_variable_mean)

Naive estimator: 2.9167507617889594
Bootstrap estimator: 4.976083767421245
True value:  5.1273089485139325
